In [3]:
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv(r"cleaned_data.csv")
print(df.head())

         Order_ID  Order_Date   Ship_Date    Ship_Mode Customer_ID  \
0  ORD-2023-00001  2023-01-01  02-01-2023  First Class      C-0070   
1  ORD-2023-00001  2023-01-01  02-01-2023  First Class      C-0070   
2  ORD-2023-00002  2023-01-01  01-01-2023     Same Day      C-0016   
3  ORD-2023-00002  2023-01-01  01-01-2023     Same Day      C-0016   
4  ORD-2023-00002  2023-01-01  01-01-2023     Same Day      C-0016   

     Customer_Name Market_Segment   Region           State    City Product_ID  \
0  Kavita Banerjee       Consumer     West       Rajasthan  Jaipur     P-1052   
1  Kavita Banerjee       Consumer     West       Rajasthan  Jaipur     P-1111   
2       Neha Joshi       Consumer  Central  Madhya Pradesh  Bhopal     P-1053   
3       Neha Joshi       Consumer  Central  Madhya Pradesh  Bhopal     P-1092   
4       Neha Joshi       Consumer  Central  Madhya Pradesh  Bhopal     P-1023   

          Category Sub_Category        Product_Name  Unit_Price  Quantity  \
0  Office Suppl

In [5]:
df.shape

(9872, 19)

In [6]:
#Convert the date column
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

In [8]:
# Create the new column
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df ["Order_Date"].dt.quarter

df[["Order_Date", "Year", "Month", "Month_Name", "Quarter"]].head()


,Order_Date,Year,Month,Month_Name,Quarter
0,2023-01-01,2023,1,January,1
1,2023-01-01,2023,1,January,1
2,2023-01-01,2023,1,January,1
3,2023-01-01,2023,1,January,1
4,2023-01-01,2023,1,January,1


In [12]:
#Calculate Profit Margin
df["Profit_Margin"] = (df["Profit"] / df["Sales"]) * 100 # Column created
df[["Sales", "Profit", "Profit_Margin"]].head()

,Sales,Profit,Profit_Margin
0,1478.4,385.89,26.101867
1,7453.4,640.92,8.599029
2,5400.0,2084.67,38.605000
3,3068.5,582.26,18.975395
4,1600.0,477.85,29.865625


In [17]:
# Calculate the order value --Order Value = Quantity × Unit Price
df["Order_Value"] = df["Quantity"] * df["Unit_Price"]
df[["Order_ID", "Unit_Price", "Quantity", "Sales", "Order_Value"]].head()

,Order_ID,Unit_Price,Quantity,Sales,Order_Value
0,ORD-2023-00001,840,2,1478.4,1680
1,ORD-2023-00001,4490,2,7453.4,8980
2,ORD-2023-00002,900,6,5400.0,5400
3,ORD-2023-00002,3230,1,3068.5,3230
4,ORD-2023-00002,400,4,1600.0,1600


In [ ]:
# Customer Segmentation  -- To classify according to their total spendings

# per customer -- Because 1 customer may place multiple orders
customer_sales = df.groupby("Customer_ID")["Sales"].sum()

In [20]:
# Assigning the Customer segment
customer_segment = pd.cut(
    customer_sales,
    bins=[-float("inf"), 20000, 50000, float("inf")],
    labels=["Low Value", "Medium Value", "High Value"]
)

In [ ]:
df["Customer_Segment"] = df["Customer_ID"].map(customer_segment) # segment added to dataset
df[["Customer_ID", "Sales", "Customer_Segment"]].head(10)

,Customer_ID,Sales,Customer_Segment
0,C-0070,1478.4,High Value
1,C-0070,7453.4,High Value
2,C-0016,5400.0,High Value
3,C-0016,3068.5,High Value
4,C-0016,1600.0,High Value
5,C-0115,27684.8,High Value
6,C-0066,3292.2,High Value
7,C-0066,9006.0,High Value
8,C-0016,1053.0,High Value
9,C-0044,216.0,High Value


In [23]:
customer_segment.value_counts()

Sales
High Value      474
Low Value       181
Medium Value     99
Name: count, dtype: int64

In [ ]:
#Create at Least 5 Additional Analytical Columns
 
 #1 Sales Category
df["Sales_Category"] = pd.cut(
    df["Sales"],
    bins=[-float("inf"), 1000, 5000, float("inf")],    #Sales below ₹1,000 → Low Sales
    labels=["Low Sales", "Medium Sales", "High Sales"] #Sales from ₹1,000 to below ₹5,000 → Medium Sales
 )                                                     #Sales of ₹5,000 and above → High Sales

df["Sales_Category"].value_counts()

Sales_Category
High Sales      3944
Medium Sales    3702
Low Sales       2226
Name: count, dtype: int64

In [ ]:
#2 Profit Category
df["Profit_Category"] = pd.cut(
    df["Profit"],
    bins=[-float("inf"), 0, 1000, float("inf")],  #Profit below ₹0 → Loss
    labels=["Loss", "Low Profit", "High Profit"], #Profit from ₹0 to below ₹1,000 → Low Profit
    right=False                                   #Profit of ₹1,000 and above → High Profit
)
df["Profit_Category"].value_counts()

Profit_Category
Low Profit     5381
High Profit    3350
Loss           1141
Name: count, dtype: int64

In [ ]:
#3 Order Size --Classify order quantities
df["Order_Size"] = pd.cut(
    df["Quantity"],                                         #Quantity of 1–2 → Small Order
    bins=[0, 2, 5, float("inf")],                           #Quantity of 3–5 → Medium Order
    labels=["Small Order", "Medium Order", "Large Order"]   #Quantity above 5 → Large Order
)
df["Order_Size"].value_counts()

Order_Size
Small Order     5296
Medium Order    3494
Large Order     1082
Name: count, dtype: int64

In [ ]:
#4 Discount Band  --Categorize the discount applied to a product.

df["Discount_Band"] = pd.cut(
    df["Discount"],
    bins=[-0.000001, 0, 0.15, float("inf")],                 #Discount = 0 → No Discount
    labels=["No Discount", "Low Discount", "High Discount"]  #Discount greater than 0 and up to 0.15 → Low Discount
)                                                            #Discount above 0.15 → High Discount
df["Discount_Band"].value_counts()

Discount_Band
Low Discount     3939
No Discount      3896
High Discount    2037
Name: count, dtype: int64

In [ ]:
#5 Product Performance --Classify products according to their sales and profit
df["Product_Performance"] = "Low Performer"

df.loc[
    (df["Sales"] >= 5000) & (df["Profit"] > 0),  #Sales of ₹5,000 or more and positive profit → High Performer
    "Product_Performance"
] = "High Performer"

df.loc[
    (df["Sales"] < 5000) & (df["Profit"] > 0),  #Positive profit but sales below ₹5,000 → Average Performer
    "Product_Performance"                       #Zero or negative profit → Low Performer
] = "Average Performer"
df["Product_Performance"].value_counts()

Product_Performance
Average Performer    5665
High Performer       3066
Low Performer        1141
Name: count, dtype: int64

In [30]:
df.head()

,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Market_Segment,Region,State,City,...,Month_Name,Quarter,Profit_Margin,Order_Value,Customer_Segment,Sales_Category,Profit_Category,Order_Size,Discount_Band,Product_Performance
0,ORD-2023-00001,1680,02-01-2023,First Class,C-0070,Kavita Banerjee,Consumer,West,Rajasthan,Jaipur,...,January,1,26.101867,1680,High Value,Medium Sales,Low Profit,Small Order,Low Discount,Average Performer
1,ORD-2023-00001,8980,02-01-2023,First Class,C-0070,Kavita Banerjee,Consumer,West,Rajasthan,Jaipur,...,January,1,8.599029,8980,High Value,High Sales,Low Profit,Small Order,High Discount,High Performer
2,ORD-2023-00002,5400,01-01-2023,Same Day,C-0016,Neha Joshi,Consumer,Central,Madhya Pradesh,Bhopal,...,January,1,38.605000,5400,High Value,High Sales,High Profit,Large Order,No Discount,High Performer
3,ORD-2023-00002,3230,01-01-2023,Same Day,C-0016,Neha Joshi,Consumer,Central,Madhya Pradesh,Bhopal,...,January,1,18.975395,3230,High Value,Medium Sales,Low Profit,Small Order,Low Discount,Average Performer
4,ORD-2023-00002,1600,01-01-2023,Same Day,C-0016,Neha Joshi,Consumer,Central,Madhya Pradesh,Bhopal,...,January,1,29.865625,1600,High Value,Medium Sales,Low Profit,Medium Order,No Discount,Average Performer


In [ ]:
# Perform Basic Analysis

df.groupby("Year")["Sales"].sum().round(2) # Total Sales of each Category
df.groupby("Quarter")["Sales"].sum().round(2) #Total sales by quarter Which Quarter generated high sales
df.groupby("Category")["Profit"].sum().round(2) #Identifies the contribution of each product category to total profit
df.groupby("Customer_Segment", observed=True)["Sales"].sum().round(2) #compares total sales generated by High Value, Medium Value, and Low Value customers
df.groupby("Discount_Band", observed=True)["Profit"].sum().round(2) #This compares profit across different discount categories



Customer_Segment
Low Value         1512089.3
Medium Value      3423882.0
High Value      122195426.4
Name: Sales, dtype: float64

In [35]:
df.to_csv("transformed_retail_sales.csv", index=False)